In [1]:
import os
import torch
import pandas as pd

# Import the pre-built utilities from your repo structure
from mimic_clip import (
    ExperimentConfig,
    build_dataloaders,
    load_clip,
    run_retrieval_eval
)

def evaluate_model_checkpoint(checkpoint_path: str, loss_type: str = "soft", text_field: str = "text", embeddings_tag: str = None, batch_size: int = 128):
    """
    Loads a specific CLIP checkpoint and calculates its validation retrieval metrics.
    
    Args:
        checkpoint_path (str): Path to the trained model directory.
        loss_type (str): 'soft' or 'hard' depending on how the model was trained.
        text_field (str): The column the model evaluates on ('text', 'findings_clean', 'impression_clean').
        embeddings_tag (str): The embeddings tag required for soft-CLIP (e.g., 'biomedvlp_text').
        batch_size (int): Batch size for evaluation.
        
    Returns:
        dict: A nested dictionary containing image_to_text and text_to_image metrics.
    """
    print(f"\n" + "="*60)
    print(f"Starting evaluation for: {checkpoint_path}")
    print(f"Params: loss_type={loss_type}, text_field={text_field}, embeddings_tag={embeddings_tag}")
    print("="*60)
    
    # 1. Build configuration for data pipeline lookup (added embeddings_tag)
    config = ExperimentConfig(
        loss_type=loss_type,
        text_field=text_field,
        embeddings_tag=embeddings_tag,
        batch_size=batch_size,
        num_workers=4
    )
    config = config.finalize()
    
    # 2. Build the validation dataloader using the repo infrastructure
    _, val_loader, df_val = build_dataloaders(config, mode="eval")
    
    # 3. Load weights from the specific checkpoint path
    if not os.path.exists(checkpoint_path):
        print(f"Error: Checkpoint path does not exist: {checkpoint_path}")
        return None
        
    model, processor, device = load_clip(checkpoint=checkpoint_path)
    
    # 4. Run retrieval evaluation suite
    metrics = run_retrieval_eval(
        model=model, 
        processor=processor, 
        val_loader=val_loader, 
        df_val=df_val, 
        device=device
    )
    
    return metrics

/home/omertole/.conda/envs/aimd/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# List of checkpoints to evaluate with their configuration blueprints
models_to_evaluate = [
    {
        "path": "checkpoints/soft/soft_text_bioclinicalbert_text_a0.5_t0.1",
        "loss_type": "soft",
        "text_field": "text", 
        "embeddings_tag": "bioclinicalbert_text"
    },
    {
        "path": "checkpoints/soft/soft_text_biomedvlp_text_a0.5_t0.1",
        "loss_type": "soft",
        "text_field": "text",
        "embeddings_tag": "biomedvlp_text"
    },
    {
        "path": "checkpoints/soft/soft_findings_clean_bioclinicalbert_findings_clean_a0.5_t0.1",
        "loss_type": "soft",
        "text_field": "text",
        "embeddings_tag": "bioclinicalbert_findings_clean"
    },
    {
        "path": "checkpoints/soft/soft_findings_clean_biomedvlp_findings_clean_a0.5_t0.1",
        "loss_type": "soft",
        "text_field": "text",
        "embeddings_tag": "biomedvlp_findings_clean"
    },
    {
        "path": "checkpoints/soft/soft_impression_clean_bioclinicalbert_impression_clean_a0.5_t0.1",
        "loss_type": "soft",
        "text_field": "text",
        "embeddings_tag": "bioclinicalbert_impression_clean"
    },
    {
        "path": "checkpoints/soft/soft_impression_clean_biomedvlp_impression_clean_a0.5_t0.1",
        "loss_type": "soft",
        "text_field": "text",
        "embeddings_tag": "biomedvlp_impression_clean"
    },
]

# Dictionary to accumulate all your extracted metrics results
all_results = {}

for m in models_to_evaluate:
    res = evaluate_model_checkpoint(
        checkpoint_path=m["path"],
        loss_type=m["loss_type"],
        text_field=m["text_field"], 
        embeddings_tag=m["embeddings_tag"]
    )
    
    if res is not None:
        all_results[m["path"]] = res

# Example of accessing specific scores afterwards
print("\nSummary of Recall@1 Scores:")
for path, metrics in all_results.items():
    i2t_r1 = metrics["image_to_text"]["recall_at_1"]
    t2i_r1 = metrics["text_to_image"]["recall_at_1"]
    print(f"-> {path} | Image-to-Text R@1: {i2t_r1:.2f}% | Text-to-Image R@1: {t2i_r1:.2f}%")


Starting evaluation for: checkpoints/soft/soft_text_bioclinicalbert_text_a0.5_t0.1
Params: loss_type=soft, text_field=text, embeddings_tag=bioclinicalbert_text


Loading weights: 100%|██████████| 398/398 [00:00<00:00, 2228.51it/s]


CLIP model loaded from checkpoints/soft/soft_text_bioclinicalbert_text_a0.5_t0.1 on device: cuda
Extracting features for the Validation Set for Evaluation...


Validation Batches: 100%|██████████| 23/23 [00:07<00:00,  2.96it/s]



--- Calculating Retrieval Metrics (Study-Level) ---

[Image-to-Text Retrieval (Study-Level)]
Recall@1 : 4.65% | Recall@5 : 11.29% | Recall@10: 17.16% | Median R : 78.0 | MRR: 0.0895

[Text-to-Image Retrieval (Study-Level)]
Recall@1 : 4.93% | Recall@5 : 14.54% | Recall@10: 22.75% | Median R : 47.5 | MRR: 0.1103


Starting evaluation for: checkpoints/soft/soft_text_biomedvlp_text_a0.5_t0.1
Params: loss_type=soft, text_field=text, embeddings_tag=biomedvlp_text


...


--- Calculating Retrieval Metrics (Study-Level) ---

[Image-to-Text Retrieval (Study-Level)]
Recall@1 : 1.29% | Recall@5 : 3.46% | Recall@10: 6.25% | Median R : 265.0 | MRR: 0.0324

[Text-to-Image Retrieval (Study-Level)]
Recall@1 : 1.26% | Recall@5 : 4.51% | Recall@10: 7.86% | Median R : 245.0 | MRR: 0.0363


Starting evaluation for: checkpoints/soft/soft_impression_clean_biomedvlp_impression_clean_a0.5_t0.1
Params: loss_type=soft, text_field=text, embeddings_tag=biomedvlp_impression_clean


Loading weights: 100%|██████████| 398/398 [00:00<00:00, 1005.07it/s]


CLIP model loaded from checkpoints/soft/soft_impression_clean_biomedvlp_impression_clean_a0.5_t0.1 on device: cuda
Extracting features for the Validation Set for Evaluation...


Validation Batches: 100%|██████████| 23/23 [00:06<00:00,  3.71it/s]



--- Calculating Retrieval Metrics (Study-Level) ---

[Image-to-Text Retrieval (Study-Level)]
Recall@1 : 2.24% | Recall@5 : 6.32% | Recall@10: 10.48% | Median R : 178.0 | MRR: 0.0515

[Text-to-Image Retrieval (Study-Level)]
Recall@1 : 1.50% | Recall@5 : 7.16% | Recall@10: 12.54% | Median R : 202.5 | MRR: 0.0514


Summary of Recall@1 Scores:
-> checkpoints/soft/soft_text_bioclinicalbert_text_a0.5_t0.1 | Image-to-Text R@1: 4.65% | Text-to-Image R@1: 4.93%
-> checkpoints/soft/soft_text_biomedvlp_text_a0.5_t0.1 | Image-to-Text R@1: 5.10% | Text-to-Image R@1: 5.84%
-> checkpoints/soft/soft_findings_clean_bioclinicalbert_findings_clean_a0.5_t0.1 | Image-to-Text R@1: 2.27% | Text-to-Image R@1: 2.45%
-> checkpoints/soft/soft_impression_clean_bioclinicalbert_impression_clean_a0.5_t0.1 | Image-to-Text R@1: 1.29% | Text-to-Image R@1: 1.26%
-> checkpoints/soft/soft_impression_clean_biomedvlp_impression_clean_a0.5_t0.1 | Image-to-Text R@1: 2.24% | Text-to-Image R@1: 1.50%
